# 12 · S3 vs. HDFS Trade-offs (Caso D)

**Teoria**: docs/08-spark-e-armazenamento-objetos.md

**Pré-requisito**: este notebook é mais pesado que os outros — ele compara
os Casos C e D lado a lado, então precisa de **ambos**
`make up-hadoop` e `make up-s3` executando simultaneamente. Feche outros
apps que consomem muitos recursos; este é o único momento do laboratório onde você está executando
~10 containers de uma vez.

---

🎯 **Objetivo deste laboratório:**

Até agora, você usou HDFS e S3 em notebooks separados. Este laboratório coloca
**ambos lado a lado** para que você possa comparar diretamente as diferenças
de desempenho em operações específicas.

📌 **O que vamos medir:**
1. **Custo de commit**: HDFS usa `rename()` O(1); S3 precisa de `copy + delete` O(N)
2. **Localidade de dados**: HDFS permite que executores leiam do nó mais próximo;
   S3 não oferece localidade (toda leitura é remota)

⚠️ **Atenção**: neste laboratório dockerizado, tanto HDFS quanto S3 estão na
**mesma rede Docker**. As diferenças que você medir serão menores que em
ambientes reais (bare metal ou nuvem), mas o **padrão relativo** se mantém.

> 💡 **Dica**: prepare os dois terminais: um com `make up-hadoop` rodando, outro
> com `make up-s3` rodando. Verifique `make status` em ambos antes de começar.

In [ ]:
import sys
import time

# Adiciona scripts/ ao path do Python
sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, get_yarn_session, layer_path

# Cria DUAS SparkSessions simultâneas:
# 1. yarn_spark → conectada ao YARN (Caso C) — lê/escreve no HDFS via webhdfs://
# 2. connect_spark → conectada via Spark Connect (Caso D) — lê/escreve no S3 via s3a://
yarn_spark = get_yarn_session("12-tradeoffs-yarn")
connect_spark = get_connect_session("12-tradeoffs-connect")

print("Both sessions created:")
print(f"  YARN session (HDFS):    {yarn_spark}")
print(f"  Connect session (S3):   {connect_spark}")

### 🎯 Duas sessões, dois mundos

Agora temos duas SparkSessions ativas simultaneamente:

| Sessão | Cluster | Armazenamento | URI |
|---|---|---|---|
| `yarn_spark` | YARN (resourcemanager + nodemanagers) | HDFS | `webhdfs://localhost:14000/...` |
| `connect_spark` | Spark Standalone (master + worker) | RustFS (S3) | `s3a://.../...` |

🧠 **Por que duas sessões?**
- Cada sessão aponta para um cluster diferente
- Os clusters (YARN e Standalone) operam de forma independente
- Podemos alternar entre eles na mesma célula
- Isso permite comparações **justas**: mesmo hardware, mesmo dataset, mesmo código

> 📌 **Cada sessão é independente**: uma não interfere na outra. Você pode
> submeter Jobs a ambas simultaneamente.

## Custo de commit: baseado em rename (HDFS) vs. baseado em cópia (S3)

Uma das diferenças mais significativas entre HDFS e Object Stores é a operação
**rename** / **commit**.

📌 **O problema:**
- O Spark escreve resultados em um diretório **temporário** (`_temporary/`)
- Ao finalizar, ele **renomeia** o diretório temporário para o destino final

🔧 **Como cada armazenamento lida com isso:**
| Operação | HDFS | S3 (Object Store) |
|---|---|---|
| `rename()` | Troca de ponteiro de metadados O(1) — instantâneo | Não existe! Requer **cópia + exclusão** de cada objeto |
| Commit | Quase gratuito, independente do tamanho | Custa O(n) — mais arquivos = mais requisições |
| Sobrescrita | Remove diretório e recria (metadados) | Lista + apaga + recria todos os objetos |

> Experimentos em nuvem real mostram que o S3 pode ser **5-20× mais lento**
> que o HDFS em operações de commit, especialmente com muitas partições pequenas.

Vamos medir isso na prática:

In [ ]:
# --- HDFS: commit via rename O(1) ---
# Lê os dados de vendas do HDFS e sobrescreve em um diretório temporário
start = time.perf_counter()
(yarn_spark.read.parquet("webhdfs://localhost:14000/datalake/bronze/vendas")
    .write.mode("overwrite")
    .parquet("webhdfs://localhost:14000/datalake/tmp/commit_test"))
hdfs_seconds = time.perf_counter() - start

# --- S3: commit via copy+delete O(n) ---
# A mesma operação, mas no RustFS via S3A.
# O Spark escreve em s3a://gold/_temporary/... e depois "renomeia" copiando
# cada arquivo + deletando o original — muito mais custoso.
start = time.perf_counter()
(connect_spark.read.parquet(layer_path("s3", "bronze", "vendas"))
    .write.mode("overwrite")
    .parquet("s3a://gold/commit_test"))
s3_seconds = time.perf_counter() - start

# Resultados
print(f"HDFS overwrite-write: {hdfs_seconds:.2f}s")
print(f"S3   overwrite-write: {s3_seconds:.2f}s")

# Análise
print("\nThe gap widens with output size/partition count — HDFS's rename is")
print("O(1) regardless of file count; S3's commit is O(files) copy+delete.")

### 📊 Análise dos tempos de commit

Você deve ter observado que o HDFS foi significativamente mais rápido que o S3
nesta operação.

📌 **Por que o S3 é mais lento?**

O `write.mode("overwrite")` no S3 executa:
1. Escreve arquivos em `s3a://gold/_temporary/0/.../`
2. Lista TODOS os objetos em `s3a://gold/commit_test/`
3. Para cada objeto, faz uma requisição `DeleteObject`
4. Lista os objetos em `s3a://gold/_temporary/`
5. Para cada objeto temporário, faz uma requisição `CopyObject` + `DeleteObject`

Em contraste, no HDFS:
1. Escreve arquivos em `/datalake/tmp/_temporary/`
2. Executa `rename(/datalake/tmp/_temporary/.../commit_test, /datalake/tmp/commit_test)`
3. Uma única operação de metadados — instantânea

🧠 **Implicações práticas:**
- Jobs Spark que produzem **muitas partições pequenas** sofrem mais no S3
- A opção `spark.sql.sources.commitProtocolClass` permite escolher protocolos
  de commit otimizados para S3 (como o "Committer" do Apache Iceberg / Delta Lake)
- Para workloads pesados de escrita, o HDFS ainda leva vantagem significativa

> 💡 **Nota sobre a medição**: neste laboratório, ambos cruzam a rede Docker,
> então a diferença absoluta é menor que em ambientes reais. Mas a **proporção**
> (HDFS muito mais rápido em rename) se mantém.

## Localidade: uma viagem de rede de qualquer forma, mas formatos diferentes

Nem os executores do Caso C (puxando dos DataNodes pela rede Docker)
nem os do Caso D (puxando do RustFS pela rede Docker) obtêm leituras verdadeiramente "locais
do disco" neste laboratório — a rede Docker significa que tudo cruza uma
rede virtual independentemente.

🎯 **No entanto, o conceito de localidade ainda importa:**

**Em um cluster HDFS bare-metal real:**
- Os DataNodes executam NAS MESMAS máquinas físicas que os NodeManagers
- O Spark agenda Tasks para executar no mesmo nó onde o bloco reside
- Isso é **data locality** — o dado está local no disco da máquina que processa
- Zero tráfego de rede para leitura de dados

**Em um Object Store (S3/RustFS):**
- Os dados ficam em servidores separados
- Não importa onde o executor roda — ele sempre faz requisição HTTP
- Isso é **não-localidade estrutural** — por design, o armazenamento é separado
  do processamento

📌 **Trade-off:**
- HDFS: melhor desempenho, mas acoplamento armazenamento-processamento
- S3: pior latência, mas **desacoplamento total** — escala armazenamento e
  processamento independentemente

> 💡 Em nuvem, esse trade-off é ainda mais relevante: você pode "ligar" um cluster
> Spark só quando precisa processar, ler do S3, processar, e desligar — pagando
> apenas pelo processamento. No HDFS, o cluster precisa ficar ligado 24/7.

### ⚠️ Atenção sobre localidade neste laboratório

Neste ambiente dockerizado:
- **HDFS**: os executores YARN e os DataNodes são containers diferentes
  → toda leitura cruza a rede Docker
- **S3/RustFS**: os Workers e o RustFS são containers diferentes
  → toda leitura cruza a rede Docker

📌 **Resultado**: a diferença de localidade será MUITO menor do que em um
cluster real. O objetivo aqui é **entender o conceito**, não medir
benchmarks absolutos.

Para ver a diferença real de localidade, você precisaria de um cluster
bare-metal ou em nuvem com dados em escala (≥ 10GB).

> 💡 **Leitura recomendada**: docs/08-spark-e-armazenamento-objetos.md para
> uma discussão aprofundada dos trade-offs com exemplos de números reais.

In [ ]:
# Finaliza ambas as SparkSessions
# Cada uma libera seus recursos de forma independente
yarn_spark.stop()
connect_spark.stop()

print("✅ Ambas as sessões encerradas.")
print("📌 Recap: HDFS vence em commit (rename O(1) vs copy+delete O(N)).")
print("📌 S3 vence em elasticidade e desacoplamento (escala independente).")
print("📌 A escolha depende da sua workload: muitas escritas → HDFS; dados massivos + elasticidade → S3.")